# Credit Scoring — End-to-End (UCI Credit Card)
Notebook completo: download de dados, limpeza, EDA, modelagem (Logistic/Tree/RF/XGB), explicabilidade (SHAP) e export para dashboard.

## 0) Setup

In [ ]:
# Se estiver no Colab, descomente para garantir dependências:
# !pip -q install pandas numpy scikit-learn xgboost imbalanced-learn shap matplotlib seaborn plotly openpyxl

import os, pathlib, warnings
warnings.filterwarnings('ignore')

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve, auc

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

ROOT = pathlib.Path('.')
DATA = ROOT/'data'; DATA.mkdir(exist_ok=True, parents=True)
DASH = ROOT/'dashboards'/'export'; DASH.mkdir(exist_ok=True, parents=True)

print('DATA dir:', DATA.resolve())
print('DASH dir:', DASH.resolve())


## 1) Aquisição de Dados (UCI)

In [ ]:
import urllib.request, sys

urls = [
    'https://raw.githubusercontent.com/ArinB/MSDS-422/master/Week5/UCI_Credit_Card.csv',
    'https://raw.githubusercontent.com/kirenz/datasets/master/uci_credit_card.csv'
]
csv_path = DATA/'UCI_Credit_Card.csv'
downloaded = False
for u in urls:
    try:
        print('Baixando:', u)
        urllib.request.urlretrieve(u, csv_path)
        downloaded = True
        break
    except Exception as e:
        print('Falha:', u, e)
if not downloaded:
    raise RuntimeError('Faça upload manual do CSV para data/UCI_Credit_Card.csv')

print('Arquivo salvo em:', csv_path)


## 2) Leitura, limpeza e split

In [ ]:
import importlib.util, sys
spec = importlib.util.spec_from_file_location('utils', str(ROOT/'src'/'utils.py'))
utils = importlib.util.module_from_spec(spec); sys.modules['utils'] = utils; spec.loader.exec_module(utils)

df = utils.robust_read_credit_card_csv(str(csv_path))
df = utils.downcast_numeric(df)

target = 'target_default'
X = df.drop(columns=[target])
y = df[target].astype(int)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent'))]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
print('Treino:', X_train.shape, 'Teste:', X_test.shape, 'Positivos Teste:', y_test.mean().round(3))


## 3) EDA rápida

In [ ]:
y.value_counts(normalize=True).mul(100).plot(kind='bar', figsize=(4,3), title='Distribuição do Target (%)')
plt.show()

corr = df[num_cols+[target]].corr()[[target]].sort_values(by=target, ascending=False)
plt.figure(figsize=(6,6)); sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues'); plt.title('Correlação com Target'); plt.show()

if 'limit_bal' in df.columns:
    sns.kdeplot(data=df, x='limit_bal', hue=target, common_norm=False)
    plt.title('Limite de crédito por classe'); plt.show()


## 4) Modelagem (Logistic, Tree, RF, XGB) + oversampling

In [ ]:
models = {
    'logistic': LogisticRegression(max_iter=1000),
    'tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'rf': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    'xgb': XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                         random_state=42, reg_lambda=1.0, n_jobs=-1, eval_metric='logloss', tree_method='hist')
}

def evaluate(pipe):
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:,1]
    preds = (proba >= 0.5).astype(int)
    return {
        'auc': roc_auc_score(y_test, proba),
        'f1': f1_score(y_test, preds),
        'precision': precision_score(y_test, preds),
        'recall': recall_score(y_test, preds)
    }, proba

import pandas as pd
rows = []; best_auc=-1; best=None; best_name=None; best_proba=None
for name, clf in models.items():
    pipe = ImbPipeline([('pre', pre), ('ros', RandomOverSampler(random_state=42)), ('clf', clf)])
    m, proba = evaluate(pipe)
    rows.append({'model':name, **m})
    if m['auc']>best_auc: best_auc=m['auc']; best=pipe; best_name=name; best_proba=proba

res = pd.DataFrame(rows).sort_values('auc', ascending=False)
print(res)
res.to_csv(DASH/'model_metrics.csv', index=False)

fpr, tpr, _ = roc_curve(y_test, best_proba)
plt.figure(figsize=(5,4)); plt.plot(fpr, tpr, label=f'ROC {best_name}'); plt.plot([0,1],[0,1],'--', c='gray'); plt.legend(); plt.title('Curva ROC'); plt.show()

scored = X_test.copy(); scored['y_true']=y_test.values; scored['score_proba']=best_proba
scored.to_csv(DASH/'scored_test_samples.csv', index=False)
print('Exports salvos em:', (DASH/'model_metrics.csv').resolve(), (DASH/'scored_test_samples.csv').resolve())


## 5) SHAP (explicabilidade) — amostra

In [ ]:
import shap
best.fit(X_train, y_train)
Xtr = best.named_steps['pre'].transform(X_test)
explainer = shap.TreeExplainer(best.named_steps['clf'])
sv = explainer.shap_values(Xtr[:200])
print('SHAP calculado (amostra de 200).')


## 6) Próximos passos

- Ajustar thresholds de decisão conforme risco/retorno do negócio.
- Conectar CSVs ao Looker Studio/Power BI para dashboard.
- Exportar o modelo com `scripts/train_export.py` e servir na API FastAPI.